In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder \
    .appName("DeltaLakePractice") \
    .getOrCreate()

In [0]:
path = "/tmp/delta_demo"

df = spark.read.format("delta").load(path)

print("Current Data")


delta_table = DeltaTable.forPath(spark, path)

Current Data


In [0]:
incremental_data = [
    (1, "Alice", 3000),
    (2, "Bob", 4000),
    (61, "NewCust1", 2000),
    (62, "NewCust2", 3500)
]

inc_df = spark.createDataFrame(incremental_data, ["id", "name", "amount"])

In [0]:
new_data = [(7, "Sam", 2500), (8, "Tom", 3200)]
new_df = spark.createDataFrame(new_data, ["id", "name", "amount"])

new_df.write.format("delta").mode("append").saveAsTable("delta_demo")

print("After Insert")
spark.table("delta_demo").show()

After Insert
+---+----+------+
| id|name|amount|
+---+----+------+
|  7| Sam|  2500|
|  8| Tom|  3200|
|  7| Sam|  2500|
|  8| Tom|  3200|
+---+----+------+



In [0]:
delta_table = DeltaTable.forName(spark, "delta_demo")

delta_table.update(
    condition="id = 1",
    set={"amount": "5000"}
)

print("After Update")
spark.table("delta_demo").show()

After Update
+---+----+------+
| id|name|amount|
+---+----+------+
|  7| Sam|  2500|
|  8| Tom|  3200|
|  7| Sam|  2500|
|  8| Tom|  3200|
+---+----+------+



In [0]:
inc_df = spark.createDataFrame(incremental_data, ["id","name","amount"])

delta_table.alias("target").merge(
    inc_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set={
    "name": "source.name",
    "amount": "source.amount"
}).whenNotMatchedInsert(values={
    "id": "source.id",
    "name": "source.name",
    "amount": "source.amount"
}).execute()

print("After Merge")
spark.table("delta_demo").display()

After Merge


id,name,amount
61,NewCust1,2000
7,Sam,2500
8,Tom,3200
7,Sam,2500
8,Tom,3200
62,NewCust2,3500
1,Alice,3000
2,Bob,4000


In [0]:
inc_df_new = inc_df.withColumn("category", lit("General"))

inc_df_new.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("delta_demo")

print("Schema After Evolution")
spark.table("delta_demo").printSchema()

Schema After Evolution
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- category: string (nullable = true)



In [0]:
spark.sql("DESCRIBE HISTORY delta_demo").display(truncate=False)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
14,2026-04-14T16:17:43.000Z,70812224371903,22pa1a0467@vishnu.edu.in,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3745194996009644),bf11e537-6789-47dc-b52b-f6de0bf83ef4,0414-155437-r4dlgvv4-v2n,13,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 1361)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-14T16:17:34.000Z,70812224371903,22pa1a0467@vishnu.edu.in,MERGE,"Map(predicate -> [""(id#15813L = id#15828L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3745194996009644),e1d24d49-ab95-4317-9b9b-058f84420704,0414-155437-r4dlgvv4-v2n,12,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 4211, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2479, materializeSourceTimeMs -> 335, numTargetRowsInserted -> 4, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1009, numTargetRowsUpdated -> 0, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 980)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
12,2026-04-14T16:17:28.000Z,70812224371903,22pa1a0467@vishnu.edu.in,UPDATE,"Map(predicate -> [""(id#15676L = 1)""])",null,List(3745194996009644),87e6b162-343a-4072-bfd5-c4bb1d4f8ebb,0414-155437-r4dlgvv4-v2n,11,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 275, numDeletionVectorsUpdated -> 0, scanTimeMs -> 274, numAddedFiles -> 0, numUpdatedRows -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-14T16:17:25.000Z,70812224371903,22pa1a0467@vishnu.edu.in,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3745194996009644),775db800-1c23-435e-bba0-916359d71e7d,0414-155437-r4dlgvv4-v2n,10,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1063)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-13T16:26:56.000Z,70812224371903,22pa1a0467@vishnu.edu.in,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(3745194996009644),63193e27-827d-4ec2-9efc-e00ec0ab2d85,0413-160013-o69ezje1-v2n,9,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 0, numRemovedFiles -> 0, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1063)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-13T16:23:43.000Z,70812224371903,22pa1a0467@vishnu.edu.in,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(3745194996009644),bb90db08-35c2-4172-8ff6-943cfbd0c7b4,0413-160013-o69ezje1-v2n,8,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 7996, numRemovedFiles -> 7, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1063)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
8,2026-04-13T16:21:26.000Z,70812224371903,22pa1a0467@vishnu.edu.in,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(3745194996009644),017d7b69-0169-4554-b93b-e20d697d0d7e,0413-160013-o69ezje1-v2n,7,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes 

In [0]:
old_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("delta_demo")

print("Old Version")
old_df.show()

Old Version
+---+----+------+
| id|name|amount|
+---+----+------+
|  7| Sam|  2500|
|  8| Tom|  3200|
+---+----+------+



In [0]:
delta_table.restoreToVersion(0)

print("After Restore")
spark.table("delta_demo").display()

After Restore


id,name,amount
7,Sam,2500
8,Tom,3200
